In [215]:
import torch
import torch.nn as nn

In [216]:
text = "hello world hello machine learning pytorch is powerful learning is fun"

In [217]:
# -------------------------
# 1. Character vocabulary
# -------------------------

chars = sorted(set(text))

stoi = {ch:i for i, ch in enumerate(chars) }
itos = {i:ch for ch, i in stoi.items()}

vocab_size = len(chars)
print("vocab_size:", vocab_size)
print("stoi:", stoi)


vocab_size: 20
stoi: {' ': 0, 'a': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'l': 9, 'm': 10, 'n': 11, 'o': 12, 'p': 13, 'r': 14, 's': 15, 't': 16, 'u': 17, 'w': 18, 'y': 19}


In [218]:
# -------------------------
# 2. Prepare data
# -------------------------

BLOCK_SIZE = 4
D_MODEL = 64
VOCAB_SIZE = len(chars)
HIDDEN_SIZE = 128

data = torch.tensor([stoi[x] for x in text])


print(data.size())
print("".join(itos[x.item()] for x in data[0:4]) + "-> " + "".join(itos[data[4].item()]))

X =  []
Y =  []

for i in range(len(data) - BLOCK_SIZE):
    X.append( data[i: i + BLOCK_SIZE] )  # previous 3 characters
    Y.append(data[i + BLOCK_SIZE])   # current character

X = torch.stack(X)
Y = torch.stack(Y)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

torch.Size([70])
hell-> o
X shape: torch.Size([66, 4])
Y shape: torch.Size([66])


In [219]:
# -------------------------
# 3. MLP model (Bengio et al. 2003)
# -------------------------



class MLPModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            D_MODEL
        )

        self.fc1 = nn.Linear(
            BLOCK_SIZE * D_MODEL,  # 4 × 64 = 256
            HIDDEN_SIZE  # 128
        )

        self.fc2 = nn.Linear(
            HIDDEN_SIZE,
            D_MODEL
        )

    def forward(self, x):
        x = self.embedding(x)
        x = x.view(x.shape[0], -1)
        x = torch.relu(self.fc1(x))
        logits = self.fc2(x)
        return logits

mlp = MLPModel()

trainable_params = sum(
    p.numel() for p in mlp.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable_params)


Trainable parameters: 42432


In [230]:
# -------------------------
# 4. Training + accuracy
# -------------------------

torch.manual_seed(42)


optimizer = torch.optim.AdamW(
    mlp.parameters(),
    lr=0.001
)

for step in range(500):

    logits = mlp(X)

    loss = nn.functional.cross_entropy(
        logits,
        Y
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        preds = logits.argmax(dim=-1)
        accuracy = (preds == Y).float().mean()
        print(f"step {step} | loss {loss.item():.4f} | accuracy {accuracy.item():.2%}")

with torch.no_grad():
    preds = mlp(X).argmax(dim=-1)
    accuracy = (preds == Y).float().mean()

print(f"Final accuracy: {accuracy.item():.2%}")

step 0 | loss 0.0638 | accuracy 95.45%
step 100 | loss 0.0630 | accuracy 95.45%
step 200 | loss 0.0630 | accuracy 95.45%
step 300 | loss 0.0630 | accuracy 95.45%
step 400 | loss 0.0630 | accuracy 95.45%
Final accuracy: 95.45%


In [232]:
# -------------------------
# 5. Generate
# -------------------------
context = torch.tensor([
    [stoi["h"],
     stoi["e"],
     stoi["l"],
     stoi["l"]]
])

result = "hell"

for _ in range(20):

    logits = mlp(context) 

    probs = torch.softmax(logits, dim=-1)
  
    next_char = torch.multinomial(
        probs,
        num_samples=1
    )
    
    result += itos[next_char.item()]

    context = torch.cat(
        [context[:, 1:], next_char],
        dim=1
    )

print("\nGenerated:")
print(result)


Generated:
hello world hello machin
